# 📊 Marketing Funnel & Conversion Performance Analysis
**Dataset:** Olist Brazilian E-Commerce — Marketing Qualified Leads & Closed Deals  
**Task:** FUTURE_DS_03 (v3) | Future Interns — Data Science & Analytics  
**Objective:** Analyze the marketing funnel to identify conversion drop-offs, channel performance, and actionable growth opportunities.


## 1. Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Green & Cyan theme
BG_DARK    = '#0a1f1f'
BG_CARD    = '#0d2b2b'
ACCENT     = '#00b894'
ACCENT2    = '#00cec9'
TEXT_LIGHT = '#e2e8f0'
GRID_COLOR = '#1f3a3a'
PALETTE    = ['#00b894', '#00cec9', '#55efc4', '#81ecec', '#6c5ce7', '#a29bfe', '#fdcb6e', '#e17055']

def style_fig(fig, title=None, height=450):
    fig.update_layout(
        plot_bgcolor=BG_CARD,
        paper_bgcolor=BG_DARK,
        font=dict(color=TEXT_LIGHT, size=12),
        title=dict(text=title, font=dict(size=16, color=TEXT_LIGHT)) if title else None,
        margin=dict(l=10, r=10, t=60 if title else 10, b=10),
        height=height,
        legend=dict(bgcolor=BG_CARD, bordercolor=GRID_COLOR, borderwidth=1, font=dict(color=TEXT_LIGHT)),
        xaxis=dict(gridcolor=GRID_COLOR, zerolinecolor=GRID_COLOR, color=TEXT_LIGHT),
        yaxis=dict(gridcolor=GRID_COLOR, zerolinecolor=GRID_COLOR, color=TEXT_LIGHT),
    )
    return fig

# Load datasets
mqls = pd.read_csv('../dataset/olist_marketing_qualified_leads_dataset.csv', parse_dates=['first_contact_date'])
deals = pd.read_csv('../dataset/olist_closed_deals_dataset.csv', parse_dates=['won_date'])

print(f'MQLs shape: {mqls.shape}')
print(f'Closed Deals shape: {deals.shape}')
mqls.head(3)

MQLs shape: (8000, 4)
Closed Deals shape: (842, 14)


,mql_id,first_contact_date,landing_page_id,origin
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search


## 2. Data Cleaning & Preparation

In [2]:
# Check missing values
print('=== MQL Missing Values ===')
print(mqls.isnull().sum())
print('\n=== Deals Missing Values ===')
print(deals.isnull().sum())

# Normalize origin
mqls['origin'] = mqls['origin'].fillna('unknown').str.strip().replace('', 'unknown')

# Merge datasets to identify won leads
merged = mqls.merge(deals[['mql_id','won_date','business_segment','lead_type',
                             'lead_behaviour_profile','declared_monthly_revenue']],
                    on='mql_id', how='left')
merged['converted'] = merged['won_date'].notna().astype(int)

print(f'\nTotal MQLs: {len(mqls):,}')
print(f'Total Closed Deals: {len(deals):,}')
print(f'Overall Conversion Rate: {merged["converted"].mean()*100:.1f}%')

=== MQL Missing Values ===
mql_id                 0
first_contact_date     0
landing_page_id        0
origin                60
dtype: int64

=== Deals Missing Values ===
mql_id                             0
seller_id                          0
sdr_id                             0
sr_id                              0
won_date                           0
business_segment                   1
lead_type                          6
lead_behaviour_profile           177
has_company                      779
has_gtin                         778
average_stock                    776
business_type                     10
declared_product_catalog_size    773
declared_monthly_revenue           0
dtype: int64

Total MQLs: 8,000
Total Closed Deals: 842
Overall Conversion Rate: 10.5%


## 3. Funnel Overview

In [3]:
total_mqls = len(mqls)
total_deals = len(deals)
drop_off = total_mqls - total_deals
cr = total_deals / total_mqls * 100

fig = go.Figure(go.Funnel(
    y=['MQLs', 'Closed Deals'],
    x=[total_mqls, total_deals],
    textinfo='value+percent initial',
    marker=dict(color=[ACCENT2, ACCENT]),
    connector=dict(line=dict(color=GRID_COLOR, width=1)),
    textfont=dict(color='#ffffff', size=15)
))
fig = style_fig(fig, title=f'Marketing Funnel Overview | Conversion Rate: {cr:.1f}%', height=400)
fig.show()

print(f'Drop-off: {drop_off:,} leads ({100-cr:.1f}%) did NOT convert')

Drop-off: 7,158 leads (89.5%) did NOT convert


## 4. Conversion Rate by Marketing Channel (Origin)

In [4]:
channel_stats = merged.groupby('origin').agg(
    total_leads=('mql_id', 'count'),
    converted_leads=('converted', 'sum')
).reset_index()
channel_stats['conversion_rate'] = channel_stats['converted_leads'] / channel_stats['total_leads'] * 100
channel_stats = channel_stats[channel_stats['origin'] != 'unknown'].sort_values('conversion_rate', ascending=False)

# --- Lead Volume by Channel ---
vol_sorted = channel_stats.sort_values('total_leads', ascending=False)
fig1 = go.Figure(go.Bar(
    x=vol_sorted['origin'], y=vol_sorted['total_leads'],
    marker_color=PALETTE[:len(vol_sorted)],
    text=vol_sorted['total_leads'], textposition='outside',
    texttemplate='%{text:,}'
))
fig1.update_xaxes(tickangle=-30)
fig1 = style_fig(fig1, title='Lead Volume by Channel', height=420)
fig1.update_layout(yaxis_title='Total MQLs')
fig1.show()

# --- Conversion Rate by Channel ---
bar_colors = ['#55efc4' if v>=10 else '#fdcb6e' if v>=5 else '#e17055' for v in channel_stats['conversion_rate']]
fig2 = go.Figure(go.Bar(
    x=channel_stats['origin'], y=channel_stats['conversion_rate'],
    marker_color=bar_colors,
    text=channel_stats['conversion_rate'], textposition='outside',
    texttemplate='%{text:.1f}%'
))
avg_cr = channel_stats['conversion_rate'].mean()
fig2.add_hline(y=avg_cr, line_dash='dash', line_color=ACCENT2,
               annotation_text=f'Avg: {avg_cr:.1f}%', annotation_font_color=ACCENT2)
fig2.update_xaxes(tickangle=-30)
fig2 = style_fig(fig2, title='Conversion Rate by Channel (%)', height=420)
fig2.update_layout(yaxis_title='Conv. Rate %')
fig2.show()

print(channel_stats[['origin','total_leads','converted_leads','conversion_rate']].to_string(index=False))

           origin  total_leads  converted_leads  conversion_rate
      paid_search         1586              195        12.295082
   organic_search         2296              271        11.803136
   direct_traffic          499               56        11.222445
         referral          284               24         8.450704
           social         1350               75         5.555556
          display          118                6         5.084746
other_publicities           65                3         4.615385
            email          493               15         3.042596
            other          150                4         2.666667


## 5. Channel Conversion Map (Treemap)

In [5]:
tree_df = channel_stats.copy()
tree_df['label'] = tree_df['origin'].str.replace('_',' ').str.title()
tree_df['cr_text'] = tree_df['conversion_rate'].apply(lambda v: f'{v:.1f}% CR')

fig = go.Figure(go.Treemap(
    labels=tree_df['label'],
    parents=[''] * len(tree_df),
    values=tree_df['total_leads'],
    text=tree_df['cr_text'],
    textinfo='label+text+value',
    marker=dict(
        colors=tree_df['conversion_rate'],
        colorscale=[[0, '#e17055'], [0.5, '#fdcb6e'], [1, '#00b894']],
        line=dict(color=BG_DARK, width=2),
        colorbar=dict(title=dict(text='CR %', font=dict(color=TEXT_LIGHT)), tickfont=dict(color=TEXT_LIGHT))
    ),
    textfont=dict(size=14, color='#ffffff')
))
fig = style_fig(fig, title='Channel Size = Lead Volume | Color = Conversion Rate', height=420)
fig.show()

## 6. Monthly Conversion Trend

In [6]:
mqls['month'] = mqls['first_contact_date'].dt.to_period('M').astype(str)
deals['won_month'] = deals['won_date'].dt.to_period('M').astype(str)

monthly_mqls = mqls.groupby('month').size().reset_index(name='mqls')
monthly_won = deals.groupby('won_month').size().reset_index(name='won')
monthly_won.columns = ['month', 'won']

monthly = monthly_mqls.merge(monthly_won, on='month', how='left').fillna(0)
monthly['cr'] = monthly['won'] / monthly['mqls'] * 100
monthly = monthly.sort_values('month')

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=monthly['month'], y=monthly['mqls'], name='MQLs', marker_color=ACCENT2, opacity=0.75), secondary_y=False)
fig.add_trace(go.Bar(x=monthly['month'], y=monthly['won'], name='Won Deals', marker_color=ACCENT, opacity=0.95), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['cr'], name='Conv. Rate %', mode='lines+markers',
                          line=dict(color='#fdcb6e', width=3), marker=dict(size=8)), secondary_y=True)
fig.update_layout(barmode='overlay', xaxis_tickangle=-45)
fig.update_yaxes(title_text='Count', secondary_y=False, gridcolor=GRID_COLOR, color=TEXT_LIGHT)
fig.update_yaxes(title_text='Conv. Rate %', secondary_y=True, gridcolor=GRID_COLOR, color='#fdcb6e')
fig = style_fig(fig, title='Monthly MQL Volume vs Won Deals vs Conversion Rate', height=450)
fig.show()

## 7. Lead Behaviour Profile Analysis

In [7]:
profile_counts = deals['lead_behaviour_profile'].value_counts().head(6)

fig = go.Figure(go.Pie(
    labels=profile_counts.index, values=profile_counts.values,
    marker=dict(colors=PALETTE[:len(profile_counts)], line=dict(color=BG_DARK, width=2)),
    textinfo='percent', textfont=dict(size=14, color='#ffffff'),
    hole=0.35,
    pull=[0.03]*len(profile_counts)
))
fig = style_fig(fig, title='Won Deals by Lead Behaviour Profile', height=450)
fig.update_layout(legend=dict(orientation='v', x=1.05, y=0.5))
fig.show()

print(profile_counts)

lead_behaviour_profile
cat            407
eagle          123
wolf            95
shark           24
cat, wolf        8
eagle, wolf      3
Name: count, dtype: int64


## 8. Top Business Segments (Won Deals)

In [8]:
top_segs = deals['business_segment'].value_counts().head(10).sort_values()

fig = go.Figure(go.Bar(
    x=top_segs.values, y=top_segs.index, orientation='h',
    marker_color=[PALETTE[i % len(PALETTE)] for i in range(len(top_segs))],
    text=top_segs.values, textposition='outside'
))
fig = style_fig(fig, title='Top 10 Business Segments in Won Deals', height=450)
fig.update_layout(xaxis_title='Number of Won Deals')
fig.show()

## 9. Lead Type Conversion Analysis

In [9]:
lead_type_dist = deals['lead_type'].value_counts()

fig = go.Figure(go.Bar(
    x=lead_type_dist.index, y=lead_type_dist.values,
    marker_color=PALETTE[:len(lead_type_dist)],
    text=lead_type_dist.values, textposition='outside'
))
fig.update_xaxes(tickangle=-30)
fig = style_fig(fig, title='Won Deals by Lead Type', height=420)
fig.update_layout(yaxis_title='Count')
fig.show()

## 10. Key Metrics Summary

In [10]:
best_channel = channel_stats.loc[channel_stats['conversion_rate'].idxmax(), 'origin']
best_cr = channel_stats['conversion_rate'].max()
worst_channel = channel_stats.loc[channel_stats['conversion_rate'].idxmin(), 'origin']
worst_cr = channel_stats['conversion_rate'].min()
top_segment = deals['business_segment'].value_counts().index[0]
top_profile = deals['lead_behaviour_profile'].value_counts().index[0]

print('=' * 55)
print('        KEY METRICS SUMMARY — FUTURE_DS_03')
print('=' * 55)
print(f'  Total MQLs Generated      : {total_mqls:,}')
print(f'  Total Won / Closed Deals  : {total_deals:,}')
print(f'  Overall Conversion Rate   : {cr:.1f}%')
print(f'  Overall Drop-off Rate     : {100-cr:.1f}%')
print(f'  Best Converting Channel   : {best_channel} ({best_cr:.1f}%)')
print(f'  Worst Converting Channel  : {worst_channel} ({worst_cr:.1f}%)')
print(f'  Top Business Segment      : {top_segment}')
print(f'  Dominant Lead Profile     : {top_profile}')
print('=' * 55)

        KEY METRICS SUMMARY — FUTURE_DS_03
  Total MQLs Generated      : 8,000
  Total Won / Closed Deals  : 842
  Overall Conversion Rate   : 10.5%
  Overall Drop-off Rate     : 89.5%
  Best Converting Channel   : paid_search (12.3%)
  Worst Converting Channel  : other (2.7%)
  Top Business Segment      : home_decor
  Dominant Lead Profile     : cat
